# 00 — Master analysis: CPU Max 9480 hardware characterization

**Question.** What can one CRESCO8 Xeon CPU Max 9480 node sustain, per precision and memory configuration, and where do those limits sit relative to the vendor's peaks?

This notebook carries only the findings. Every number comes from a measured campaign (five steady one-second samples per point) and links back to its campaign id in the supporting notebooks: [platform and methodology](01_platform_and_methodology.ipynb), [memory](02_memory_hierarchy_and_bandwidth.ipynb), [NUMA and sockets](03_numa_and_socket_scaling.ipynb), [compute](04_compute_throughput.ipynb), [roofline atlas](05_roofline_atlas.ipynb), [LLM interface](06_llm_roofline_interface.ipynb).

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/hwbench').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
from hwbench import report, roofline, plotting
from hwbench.notebook import setup
ctx = setup('00_master_analysis')
thesis, ext, theory = ctx.thesis, ctx.ext, ctx.theory
table, figure, say = ctx.table, ctx.figure, ctx.say

## 1. Platform and what was measured

Two sockets × 56 cores, SMT off, 64 GiB HBM2e and 8-channel DDR5-4800 per socket. *Flat* mode exposes HBM and DDR as separate NUMA nodes; *Cache* mode makes HBM a memory-side cache in front of DDR. Flat results were taken on one dedicated node outside Slurm (the October runs went through an exclusive Slurm allocation on the same node and are labelled as such); Cache results come from Slurm allocations on other nodes.

In [ ]:
prov = (ctx.everything[ctx.everything.campaign_id.isin(
            set(thesis.campaign_id) | set(ext.campaign_id))]
        .groupby(['memory_mode', 'launch_method', 'node'])
        .agg(campaigns=('campaign_id', 'nunique'), samples=('accepted', 'size')).reset_index())
table(prov)

## 2. Compute: peak per execution path

The *efficiency at clock* column is the honest one: the cores do not run at the 1.9 GHz base clock. AVX-512 code ran near 2.5 GHz and AMX code near 2.0 GHz, so a datasheet comparison at base clock reads above 100% for FP64 and FP32.

In [ ]:
peaks = report.isa_peaks(ext, theory)
one = peaks[(peaks.scope == '1S') & (peaks.cores == 56)].groupby(['memory_mode', 'path'], as_index=False).first()
shown = one.pivot(index='path', columns='memory_mode', values='gops') / 1000
shown['ops/cycle/core (flat)'] = one[one.memory_mode == 'flat'].set_index('path').ops_per_cycle_per_core
shown['theory ops/cycle'] = one[one.memory_mode == 'flat'].set_index('path').theory_ops_per_cycle
shown['efficiency at clock (flat)'] = one[one.memory_mode == 'flat'].set_index('path').efficiency_at_clock
shown = shown.rename(columns={'cache': 'Cache TOP/s', 'flat': 'Flat TOP/s'}).reset_index()
table(shown)

In [ ]:
fig = plotting.go.Figure()
flat = one[one.memory_mode == 'flat'].set_index('path')
order = ['fp64', 'fp32', 'avx512_bf16', 'vnni_int8', 'amx_bf16', 'amx_int8']
fig.add_bar(x=order, y=[flat.loc[p, 'gops'] / 1000 for p in order], name='Measured (1 socket, 56 cores)',
            marker_color=[plotting.PATH_COLORS[p] for p in order])
fig.add_scatter(x=order, y=[theory.peak_gops(p, 56) / 1000 for p in order], mode='markers', name='Theoretical at 1.9 GHz base',
                marker=dict(symbol='line-ew', size=26, line=dict(width=3, color='#6B7480')))
plotting.style(fig, 'Peak throughput by execution path · one socket', 'Execution path', 'TOP/s (operations = multiply + add)', 520)
fig.update_yaxes(type='log')
figure(fig, 'compute_peaks')

In [ ]:
amx = flat.loc['amx_bf16']; i8 = flat.loc['amx_int8']; vb = flat.loc['avx512_bf16']; f32 = flat.loc['fp32']
say(f"**Result.** One socket sustains {i8.gops/1000:.0f} TOP/s on AMX INT8 and {amx.gops/1000:.0f} TOP/s on AMX BF16 "
    f"({amx.efficiency_at_clock:.1%} of the peak at the {amx.ghz_effective:.2f} GHz the cores actually ran). "
    f"The AVX-512 BF16 dot-product path reaches only {vb.gops/1000:.1f} TOP/s ({vb.ops_per_cycle_per_core:.0f} ops/cycle/core, about 0.5 instruction per cycle), "
    f"less than half of FP32 FMA ({f32.gops/1000:.1f} TOP/s): software that cannot use AMX gains little from BF16 weights in compute-bound regimes.")

## 3. Memory: sustained bandwidth by tier and kernel

In [ ]:
bw = report.bandwidth_summary(ext, thesis)
local = bw[(bw.locality == 'local') & (bw.socket_scope == 'single') & (bw.socket == 0) & (bw.working_set_gib == 8)]
table(local.pivot_table(index='config', columns='kernel', values='gbs').reset_index())

In [ ]:
l1 = report.hierarchy(ext)
lv, lat = report.level_summary(l1)
table(lat.pivot(index='level', columns='config', values='latency_ns').reset_index())

In [ ]:
p = local.pivot_table(index='config', columns='kernel', values='gbs')
say(f"**Result.** One socket's HBM sustains {p.loc['flat_hbm'].max():.0f} GB/s against {p.loc['flat_ddr'].max():.0f} GB/s for DDR "
    f"({p.loc['flat_hbm'].max()/p.loc['flat_ddr'].max():.1f}×). HBM trades latency for bandwidth: an idle dependent load costs "
    f"{lat[(lat.config=='flat_hbm')&(lat.level=='memory')].latency_ns.iloc[0]:.0f} ns on HBM against "
    f"{lat[(lat.config=='flat_ddr')&(lat.level=='memory')].latency_ns.iloc[0]:.0f} ns on DDR.")

## 4. Where the ceilings meet: ridge points

The ridge is the operational intensity at which a kernel stops being limited by memory and starts being limited by compute. It sets how much reuse a workload needs before extra compute helps.

In [ ]:
mem = report.bandwidth_summary(ext, thesis)
best = {}
for cfg in ['flat_hbm', 'flat_ddr', 'cache']:
    sel = mem[(mem.config == cfg) & (mem.locality == 'local') & (mem.socket_scope == 'single') & (mem.socket == 0)]
    if cfg == 'cache':
        sel = sel[sel.working_set_gib <= 16]
    best[cfg] = sel.gbs.max()
rows = []
for p in order:
    for cfg, gbs in best.items():
        rows.append(dict(path=p, memory=plotting.CONFIG_LABELS[cfg],
                         compute_TOPs=flat.loc[p, 'gops'] / 1000, bandwidth_GBs=gbs,
                         ridge_op_per_byte=roofline.ridge(flat.loc[p, 'gops'], gbs)))
table(pd.DataFrame(rows).pivot(index='path', columns='memory', values='ridge_op_per_byte').reindex(order).reset_index())

**Implication.** On HBM the ridge for AMX BF16 sits near 200 operations per byte, for AVX-512 FP64 near 8. LLM decode at small batch has an intensity of a few operations per byte and is therefore bandwidth-bound on every path; the compute ceiling only matters for prefill and large batches. The [roofline atlas](05_roofline_atlas.ipynb) draws these ceilings with measured GEMM points; [notebook 06](06_llm_roofline_interface.ipynb) shows where decode and prefill will fall.

## 5. Limits of these results

* Bytes are *modeled* (kernel traffic convention), not memory-controller counts: uncore counters are not readable without privileges. Write-allocate behaviour is therefore inferred, not observed.
* No power or energy was measured (RAPL needs root).
* SMT is off on these nodes; its effect is documented, not measured.
* Cache-mode HBM hit rates are inferred from bandwidth versus working-set size only.
* Peaks come from synthetic kernels; GEMM points show what a library achieves at LLM shapes.